In [1]:
# import required Python modules

import os
import requests
from azure.identity import AzureCliCredential
from datetime import datetime, timedelta, timezone
import azure.storage.blob
from urllib.parse import urlparse
import yaml

In [2]:
MPCPRO_APP_ID = "https://geocatalog.spatio.azure.com"
CONTAINER_URI = "https://mpcpstorageaccount.blob.core.windows.net/stac-items"
GEOCATALOG_URI = "https://geospatialdm.fmd9dgfcd2fab5hw.westeurope.geocatalog.spatio.azure.com"
API_VERSION = "2025-04-30-preview"

In [3]:
# obtain an access token
credential = AzureCliCredential()
access_token = credential.get_token(f"{MPCPRO_APP_ID}/.default")

In [4]:
import os 
import requests
import yaml
from pprint import pprint
from azure.identity import AzureCliCredential
import pystac


MPCPRO_APP_ID = "https://geocatalog.spatio.azure.com"
GEOCATALOG_URI = "https://geospatialdm.fmd9dgfcd2fab5hw.westeurope.geocatalog.spatio.azure.com"
API_VERSION = "2025-04-30-preview"


collection_id = "Nigeria-CHIRPS-1"
collection_title = f"Nigeria CHIRPS Collection" 
collection_desc = f"Collection of CHIRPS Earth observation data"


# Create spatial extent
bbox = [2.316388, 3.837669, 15.126447, 14.153350]  # placeholder, replace with actual data at a later date
spatial_extent = pystac.SpatialExtent([bbox])

# Create temporal extent, use current date time or replace with existing datetimes in stac_item
start_datetime = datetime(1981, 1, 1, 0, 0, tzinfo=timezone.utc)
temporal_extent = pystac.TemporalExtent([[start_datetime, None]])
extent = pystac.Extent(spatial=spatial_extent, temporal=temporal_extent)

# Create the STAC Collection
collection = pystac.Collection(
    id=collection_id,
    description=collection_desc,
    extent=extent,
    title=collection_title,
    license="private",
)

# Add keywords and provider
collection.keywords = ["CHIRPS", "satellite", "weather", "UCSB"]
collection.providers = [
    pystac.Provider(
        name="UCSB",
        roles=["producer", "licensor"],
        url="https://data.chc.ucsb.edu"
    )
]

collection_dict = collection.to_dict()
collection_dict['stac_version'] = '1.0.0'
collection_dict['item_assets'] = {
    "data": {
        "type": "image/tiff; application=geotiff",
        "roles": ["data"],
        "title": "CHIRPS Precipitation Data",
        "raster:bands": [
            {
                "data_type": "float32",
                "nodata": -9999,
                "unit": "mm"
            }
        ]
    }
}

In [6]:
# create a new collection with the collection api
response = requests.post(
    f"{GEOCATALOG_URI}/stac/collections",
    json=collection_dict,
    headers={"Authorization": "Bearer " + access_token.token},
    params={"api-version": API_VERSION},
)

if response.status_code == 201 or response.status_code == 200:
    print("Collection created successfully")
    pprint(response.json())
else:
    print(f"Failed to create ingestion: {response.text}")

# to-do: Better error handling and reporting; I think I have some code in the other files that handle this better

Failed to create ingestion: {"id":"6f228b81-e5cd-4e99-899e-5e40fa49bff8","status":"Pending","statusHistory":[{"status":"Pending","timestamp":"2025-11-07T10:38:21.2695391Z"}],"type":"AddCollection","creationTime":"2025-11-07T10:38:21.2695337Z","collectionId":"Nigeria-CHIRPS-1"}


In [7]:
import os
import requests
import yaml
from azure.identity import AzureCliCredential

MPCPRO_APP_ID = "https://geocatalog.spatio.azure.com"
GEOCATALOG_URI = "https://geospatialdm.fmd9dgfcd2fab5hw.westeurope.geocatalog.spatio.azure.com"
API_VERSION = "2025-04-30-preview"

COLLECTION_ID = collection_id
catalog_href = "https://mpcpstorageaccount.blob.core.windows.net/stac-items/catalog.json"

skip_existing_items = False
keep_original_assets = False
timeout_seconds = 300

In [8]:
# Obtain an access token
credential = AzureCliCredential()
access_token = credential.get_token(f"{MPCPRO_APP_ID}/.default")

In [9]:
url = f"{GEOCATALOG_URI}/inma/collections/{COLLECTION_ID}/ingestions"
body = {
    "importType": "StaticCatalog",
    "sourceCatalogUrl": catalog_href,
    "skipExistingItems": skip_existing_items,
    "keepOriginalAssets": keep_original_assets,
}

In [10]:
ing_response = requests.post(
    url,
    json=body,
    timeout=timeout_seconds,
    headers={"Authorization": f"Bearer {access_token.token}"},
    params={"api-version": API_VERSION},
)

In [11]:
if ing_response.status_code == 201:
    print("Ingestion created successfully")
    ingestion_id = ing_response.json()["id"]
    print(f"Created ingestion with ID: {ingestion_id}")
else:
    print(f"Failed to create ingestion: {ing_response.text}")

Ingestion created successfully
Created ingestion with ID: b1b6f856-377b-4a33-9161-febc18bf0929


In [12]:
geocatalog_url = GEOCATALOG_URI
runs_endpoint = (
    f"{geocatalog_url}/inma/collections/{collection_id}/ingestions/{ingestion_id}/runs"
)

wf_response = requests.post(
    runs_endpoint,
    headers={"Authorization": f"Bearer {access_token.token}"},
    params={"api-version": API_VERSION},
)

if wf_response.status_code == 201:
    print("Workflow started successfully")
else:
    print(f"Failed to create ingestion run: {wf_response.text}")

Workflow started successfully
